## MFP Arena Statistics
# Goal
To collect statistics on individual arenas for an MFP season, notably:
* Average arena duration
    - this notebook
- Individual Player Stats for each arena
    - Small sample size at the moment (week 4)
* Randomness of arena drawing
    - It seems that arenas are drawn fairly randomly

# Issues
The arenas often change names, leading to machines not possible being present in weeks before the change.


# To-DO
- 

In [29]:
import pandas as pd
import numpy as np
import io
import requests
import json
import lxml
from collections import defaultdict
from bs4 import BeautifulSoup

In [30]:
from fake_useragent import UserAgent
ua = UserAgent()
    #fallback='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_7_4) AppleWebKit/537.13 (KHTML, like Gecko) Chrome/24.0.1290.1 Safari/537.13')
ua.update()
%matplotlib inline

In [31]:
# Matchplay API URL
series_id = '2161' # Id number from the matchplay series

In [32]:
# Parameters
series_id = "2607"


In [33]:
try:
    orig_arena_time_df = pd.read_csv('arena_time_df_'+series_id+'.csv')
except:
    print('arena_time_df_'+series_id+'.csv not found, new file will be created.')

In [34]:
# orig_arena_time_df

In [35]:
series_url = 'https://matchplay.events/series/'+series_id+'/standings'

In [36]:
#papermill_description=calling_the_matchplay_api
# Call the API
r = requests.get(series_url)
data = r.json()
tournaments = data['tournament_points']
tournament_ids = [tournament['tournament_id'] for tournament in data['tournament_points']]

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
r

In [ ]:
tournament_ids

In [ ]:
tournaments = []
for tournament in tournament_ids:
    results_url = 'https://matchplay.events/data/tournaments/' + str(tournament) + '/results/csv'
    results = pd.read_csv(results_url)
    tournaments.append(results)
results = pd.concat(tournaments)


In [ ]:
arenas = results[['Round','Set','Game 1']]

In [ ]:
round_ids = arenas.Round.unique()

In [ ]:
standings_url = 'https://matchplay.events/live/series/'+series_id

In [ ]:
def get_soup(url, timeout=5):
    headers  = {'User-Agent':ua.random}
    try:
        response = requests.get(url,headers=headers)
    except:
        print("FAILED "+ url)
        return 0
    attempts = 0
    while(not response.ok):
            print((url+' failed with code: '+str(response.status_code)))
            if attempts > timeout:
                print(url+' failed with code: '+str(response.status_code))
                return BeautifulSoup('','lxml')
            response = requests.get(url)
            attempts += 1
    page = response.text
    soup = BeautifulSoup(page, 'lxml')
    return soup

In [ ]:
#papermill_description=scraping_standings_poge
standings_page = get_soup(standings_url)

In [ ]:
tournament_live_ids = []
for week in standings_page.find('tr').find_all('a')[:len(tournament_ids)]:
    tournament_live_ids.append(week['href'].split('/')[-1])

In [ ]:
tournament_live_ids

In [ ]:
from selenium import webdriver 
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
option = webdriver.ChromeOptions()
option.add_argument(' - incognito')

In [ ]:
#papermill_description=starting_chromedriver
driver = webdriver.Chrome(ChromeDriverManager().install())

In [ ]:
def go_to_round(driver, round_num):
    timestamps = []
    arenas = []
    element = driver.find_element_by_id('rounds-nav')
    all_options = element.find_elements_by_tag_name("option")
    all_options[-round_num].click()
    timestamps = [element.text for element in driver.find_elements_by_class_name('timestamp')]
    timestamps = timestamps[1::2]
    arenas = [element.text for element in driver.find_elements_by_class_name('arena')]
    matches = driver.find_elements_by_class_name('compact-game')
    
    num_players = [len(match.find_elements_by_class_name('loser')) for match in matches]
    players = [match.find_elements_by_class_name('player-name') for match in matches]
    player_lists = []
    for player_list in players:
        player_lists.append([player.text for player in player_list])
    places = [match.find_elements_by_class_name('position') for match in matches]
    place_lists = []
    for place_list in places:
        place_lists.append([place.text for place in place_list])
        
    
    
    group_numbers = [x+1 for x in range(len(matches))]
    round_number = [round_num for x in range(len(matches))][::-1]
    position_lists = [range(1,num+1) for num in num_players]
    return zip(arenas,timestamps,num_players,group_numbers,round_number,player_lists,place_lists,position_lists)

In [ ]:
def get_all_rounds(driver, live_id):
    url = 'https://matchplay.events/live/'+live_id+'/matches'
    driver.get(url)
    
    arena_times = []
    for round_num in range(1,6):
        arena_times += go_to_round(driver, round_num)
    return arena_times

In [ ]:
def get_all_weeks(driver, live_ids):
    arena_times = []
    for live_id in live_ids:
        arena_times += get_all_rounds(driver, live_id)
    return arena_times

In [ ]:
#papermill_description=scraping_all_weeks_in_season
all_times = get_all_weeks(driver, tournament_live_ids)

In [ ]:
arena_time_df = pd.DataFrame(all_times,columns=['Arena','Time','num_players','group_number','round_number','players','places','positions'])

In [ ]:
#papermill_description=adjusting_times
arena_time_df['Time'] = pd.to_datetime(arena_time_df['Time'],format='%M:%S')

In [ ]:
arena_time_df['Time'] = (arena_time_df['Time'].dt.minute*60)+arena_time_df['Time'].dt.second

In [ ]:
arena_time_df['adj_time'] = arena_time_df['Time']*(4/arena_time_df['num_players'])

In [ ]:
arena_time_df.Arena = arena_time_df.Arena.str.replace('\(PinTips\)','')

In [ ]:
#papermill_description=saving_arena_time_csv
arena_time_df.to_csv('arena_time_df_'+series_id+'.csv')

In [ ]:
# PRobable break between pulling data to csv/plot
arena_time_df[arena_time_df.group_number == 1].shape

In [ ]:
def explode(df, lst_cols, fill_value=''):
    # make sure `lst_cols` is a list
    if lst_cols and not isinstance(lst_cols, list):
        lst_cols = [lst_cols]
    # all columns except `lst_cols`
    idx_cols = df.columns.difference(lst_cols)

    # calculate lengths of lists
    lens = df[lst_cols[0]].str.len()

    if (lens > 0).all():
        # ALL lists in cells aren't empty
        return pd.DataFrame({
            col:np.repeat(df[col].values, lens)
            for col in idx_cols
        }).assign(**{col:np.concatenate(df[col].values) for col in lst_cols}) \
          .loc[:, df.columns]
    else:
        # at least one list in cells is empty
        return pd.DataFrame({
            col:np.repeat(df[col].values, lens)
            for col in idx_cols
        }).assign(**{col:np.concatenate(df[col].values) for col in lst_cols}) \
          .append(df.loc[lens==0, idx_cols]).fillna(fill_value) \
          .loc[:, df.columns]

In [ ]:
#papermill_description=exploding_into_full_match_df
full_match_df = explode(arena_time_df, ['players','places','positions'])

In [ ]:
full_match_df

In [ ]:
# str(Place) : {num_players:numpoints}
place_points_dict = { '1st' : {2:7,3:7,4:7},
                      '2nd' : {4:5,3:4,2:1},
                      '3rd' : {4:3,3:1},
                      '4th' : {4:1,3:0}
                    }

def calc_points(row):
    row['points'] = place_points_dict[row['places']][row['num_players']]
    return row

In [ ]:
#papermill_description=calculating_full_match_df_pts
full_match_df = full_match_df.apply(calc_points, axis=1)

In [ ]:
full_match_df

In [ ]:
#papermill_description=saving_full_match_df_csv
full_match_df.to_csv('mfp_league_'+series_id+'_full.csv', encoding='utf-8')